In [7]:
#Import các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [8]:
import zipfile
zf = zipfile.ZipFile('G:\MOIC\MOIC_Inventory\DataRaw\m5-forecasting-accuracy.zip')


In [9]:
sales_val = pd.read_csv(zf.open('sales_train_validation.csv'))
calendar_val = pd.read_csv(zf.open('calendar.csv'))
sell_prices_val = pd.read_csv(zf.open('sell_prices.csv'))

In [10]:
sales_val.head(5)

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [ ]:
class DemandSeries:
    def __init__(self, data, id_col:str, value_col:list[str], cate_cols:list[str]=None):
        self.data = data
        self.id_col = id_col
        self.value_col = value_col
    
    def get_series(self, id_value, drop_index = True):
        """Trả về timeseries cho id_value"""
        if not isinstance(id_value, (list, tuple, set)):
            id_value = [id_value]

        series = self.data[self.data[self.id_col].isin(id_value)].set_index(self.id_col)[self.value_col]
        return series.reset_index(drop=drop_index)


    def generate_CV2(self):
        """Tạo Coefficient of Variation series theo từng hàng (row-wise)"""
        row_std = self.data[self.value_col].std(axis=1)
        row_mean = self.data[self.value_col].mean(axis=1)
        cv2 = (row_std / row_mean) ** 2
        return cv2
    
    def generate_ADI(self):
        """Tính toán Average Demand Interval (ADI)"""
        total_periods = self.data[self.value_col].shape[1]
        non_zero_counts = (self.data[self.value_col] > 0).sum(axis=1)
        adi = total_periods / non_zero_counts
        return adi

    def classify_demand(self, adi_threshold=1.32, cv2_threshold=0.49):
        """Phân loại demand series vào 4 loại: Smooth, Erratic, Slow, Lumpy"""
        adi = self.generate_ADI()
        cv2 = self.generate_CV2()

        def classify(a, c):
            if a <= adi_threshold and c <= cv2_threshold:
                return "Smooth"
            elif a <= adi_threshold and c > cv2_threshold:
                return "Erratic"
            elif a > adi_threshold and c <= cv2_threshold:
                return "Slow"
            else:
                return "Lumpy"

        return [classify(a, c) for a, c in zip(adi, cv2)]

    


['Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Erratic',
 'Erratic',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Lumpy',
 'Erratic',
 'Lu